## Example 1: Import a notebook as a module

In [ ]:
import os
import tempfile
from pathlib import Path

import nbformat

def make_notebook(name, *cells):
    """Write <name>.ipynb into a fresh folder and make it the working directory (import_ipynb looks there)."""
    d = Path(tempfile.mkdtemp())
    nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(c) for c in cells]), d / f"{name}.ipynb")
    os.chdir(d)
    return d

import importlib
import importlib.metadata

import import_ipynb

# Example 1: after `import import_ipynb`, a notebook in the working directory imports like a .py file
assert importlib.metadata.version("import-ipynb") == "0.2.1"
make_notebook("nb_math", "def square(x):\n    return x * x", "ANSWER = square(7)")
mod = importlib.import_module("nb_math")
# (0.2.1 hands back a ModuleSpec holding the notebook's names, so the path is .origin, not __file__)
assert mod.ANSWER == 49 and mod.square(3) == 9 and mod.origin.endswith("nb_math.ipynb"), mod.origin
print("Example 1 OK:", mod.origin)

## Example 2: IPython magics in the notebook are translated

In [ ]:
import os
import tempfile
from pathlib import Path

import nbformat

def make_notebook(name, *cells):
    """Write <name>.ipynb into a fresh folder and make it the working directory (import_ipynb looks there)."""
    d = Path(tempfile.mkdtemp())
    nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(c) for c in cells]), d / f"{name}.ipynb")
    os.chdir(d)
    return d

import importlib

import import_ipynb

# Example 2: line magics run through IPython's transformer, so %time and friends do not break the import
make_notebook("nb_magic", "%time total = sum(range(10))", "result = total")
mod = importlib.import_module("nb_magic")
assert mod.result == 45, mod.result
print("Example 2 OK")

## Example 3: Markdown cells are skipped

In [ ]:
import importlib
import os
import tempfile
from pathlib import Path

import import_ipynb
import nbformat

# Example 3: only code cells run
d = Path(tempfile.mkdtemp())
nb = nbformat.v4.new_notebook()
nb.cells = [nbformat.v4.new_markdown_cell("# Title\nnot python"), nbformat.v4.new_code_cell("x = 'code ran'")]
nbformat.write(nb, d / "nb_md.ipynb")
os.chdir(d)
assert importlib.import_module("nb_md").x == "code ran"
print("Example 3 OK")

## Example 4: Notebooks import each other

In [ ]:
import os
import tempfile
from pathlib import Path

import nbformat

def make_notebook(name, *cells):
    """Write <name>.ipynb into a fresh folder and make it the working directory (import_ipynb looks there)."""
    d = Path(tempfile.mkdtemp())
    nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(c) for c in cells]), d / f"{name}.ipynb")
    os.chdir(d)
    return d

import importlib

import import_ipynb

# Example 4: a notebook can import another notebook through the same finder
d = make_notebook("nb_base", "BASE = 10")
nb = nbformat.v4.new_notebook()
nb.cells = [nbformat.v4.new_code_cell("import nb_base\nTOTAL = nb_base.BASE + 5")]
nbformat.write(nb, d / "nb_top.ipynb")
assert importlib.import_module("nb_top").TOTAL == 15
print("Example 4 OK")

## Example 5: The finder on sys.meta_path

In [ ]:
import sys

import import_ipynb

# Example 5: the import hook is a NotebookFinder appended to sys.meta_path on first import
finders = [f for f in sys.meta_path if type(f).__name__ == "NotebookFinder"]
assert len(finders) == 1, sys.meta_path
assert hasattr(import_ipynb, "NotebookLoader") and hasattr(import_ipynb, "find_notebook")
print("Example 5 OK:", finders[0])